In [0]:
import dataiku
import logging

# ==============================================================================
# Configuration & Input
# ==============================================================================
# The tag string used to identify scenarios to disable.
# Per instructions: Scenarios WITH this tag will be disabled.
IGNORE_TAG = "ignore_tag" 

# ==============================================================================
# Logging Setup
# ==============================================================================
logging.basicConfig(
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

def disable_tagged_scenarios(target_tag: str):
    """
    Enumerates all scenarios in all projects.
    If a scenario has the target_tag AND is active:
        1. Disables the scenario (auto-trigger off).
        2. Adds the 'auto_disabled' tag.
    
    Args:
        target_tag (str): The tag to look for to trigger the disable action.
    """
    try:
        # Connect to the local DSS instance
        client = dataiku.api_client()
        
        # [cite_start]Enumerate all projects [cite: 1]
        # Assumption 7 override: Scope is 'all projects', not just default.
        project_keys = client.list_project_keys()
        logger.info(f"Found {len(project_keys)} projects to scan.")

        for project_key in project_keys:
            process_project(client, project_key, target_tag)
            
    except Exception as e:
        logger.error(f"Fatal error in main execution: {e}")

def process_project(client, project_key: str, target_tag: str):
    """
    Process a single project to identify and update scenarios.
    """
    try:
        project = client.get_project(project_key)
        scenarios = project.list_scenarios()
        
        for scenario_item in scenarios:
            scenario_id = scenario_item['id']
            process_scenario(project, scenario_id, target_tag)
            
    except Exception as e:
        logger.warning(f"Failed to process project {project_key}: {e}")

def process_scenario(project, scenario_id: str, target_tag: str):
    """
    Applies logic to a single scenario:
    - Checks for tag match
    - Checks if active
    - Disables and tags if conditions met
    """
    try:
        scenario = project.get_scenario(scenario_id)
        settings = scenario.get_settings()
        
        # Check specific requirements:
        # 1. Currently has a tag that matches the ignore_tag variable
        # 2. Scenario's auto-trigger is enabled (active)
        if target_tag in settings.tags and settings.active:
            
            logger.info(f"Disabling scenario: {scenario_id} in project {project.project_key}")
            
            # Requirement: Disable the auto-trigger
            settings.active = False
            
            # Requirement: Assign tag named "auto_disabled"
            if "auto_disabled" not in settings.tags:
                settings.tags.append("auto_disabled")
            
            # Idempotent save
            settings.save()
            
    except Exception as e:
        # Log error but continue processing other scenarios
        logger.warning(f"Error processing scenario {scenario_id}: {e}")

# ==============================================================================
# Main Execution
# ==============================================================================
if __name__ == "__main__":
    disable_tagged_scenarios(IGNORE_TAG)